In [1]:
!pip install pandas scikit-learn ipywidgets --quiet

from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
import ipywidgets as widgets
import re


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.1 MB/s eta 0:00:00


In [2]:
# Define genres, names, title components
genres_list = ["Romance","Comedy","Drama","Action","Thriller","Horror","Sci-Fi",
               "Fantasy","Documentary","Family","Animation","Mystery","Crime",
               "Musical","Adventure"]

first_names = ["Alex","Jamie","Taylor","Jordan","Casey","Morgan","Chris","Pat",
               "Dana","Lee","Sam","Riley","Avery","Jesse","Robin"]

last_names  = ["Martin","Smith","Nguyen","Patel","Chen","Brown","Garcia","Müller",
               "Silva","Khan","Moyo","Dlamini","Johnson","Williams","Lopez"]

title_adjs  = ["Last","First","Dark","Misty","Bright","Lost","Secret","Hidden",
               "Final","Summer","Winter","Golden","Lonely","Crazy","Small"]
title_nouns = ["Promise","Journey","Love","Night","Day","Game","Dream","Song",
               "Road","River","House","City","Hour","Waltz","Story"]

# Functions to generate titles, plots, cast
def random_cast(n=3):
    return ", ".join(f"{random.choice(first_names)} {random.choice(last_names)}" for _ in range(n))

def generate_title():
    return f"{random.choice(title_adjs)} {random.choice(title_nouns)}"

def generate_plot(year, genre):
    verbs  = ["solve a mystery","escape a conspiracy","reconnect",
              "find true love","protect a secret","win back a family"]
    complications = ["a long-buried secret surfaces","the city's fate hangs in the balance",
                     "time is running out","a stranger arrives with news"]
    tones   = ["heartwarming","dark","lighthearted","melancholic","whimsical"]
    themes  = ["forgiveness","redemption","friendship","survival"]
    objects = ["map","diary","mysterious letter","old photograph"]
    periods = ["1990s","1980s","future","present"]

    lead1 = f"{random.choice(first_names)} {random.choice(last_names)}"
    lead2 = f"{random.choice(first_names)} {random.choice(last_names)}"
    return (f"In {year}, {lead1} and {lead2} must {random.choice(verbs)} when {random.choice(complications)}. "
            f"A {random.choice(tones)} {genre.lower()} about {random.choice(themes)} and {random.choice(objects)} "
            f"set in the {random.choice(periods)}.")

def pick_genres():
    return ";".join(random.sample(genres_list, random.choice([1,1,2])))

# Generate dataset
rows=[]
for i in range(400):
    year = random.randint(1970,2020)
    genre = random.choice(genres_list)
    rows.append({
        "title": generate_title(),
        "genre": pick_genres(),
        "cast": random_cast(),
        "plot": generate_plot(year, genre),
        "year": year
    })

df = pd.DataFrame(rows)
df.head()


,title,genre,cast,plot,year
0,Misty City,Documentary,"Robin Brown, Robin Williams, Lee Nguyen","In 1980, Avery Silva and Jordan Nguyen must wi...",1980
1,Secret Night,Romance,"Taylor Nguyen, Casey Moyo, Taylor Moyo","In 2017, Casey Smith and Jesse Chen must prote...",2017
2,Summer Road,Thriller,"Morgan Lopez, Jesse Martin, Jordan Johnson","In 1997, Taylor Brown and Jamie Garcia must fi...",1997
3,Dark Story,Comedy,"Riley Moyo, Jesse Dlamini, Avery Patel","In 1998, Avery Moyo and Sam Lopez must solve a...",1998
4,Golden House,Crime;Fantasy,"Jamie Chen, Chris Patel, Jesse Moyo","In 2003, Robin Garcia and Taylor Lopez must pr...",2003


In [3]:
# Combine text fields for TF-IDF
df["combined"] = df["title"] + " " + df["genre"] + " " + df["cast"] + " " + df["plot"] + " " + df["year"].astype(str)
vectorizer = TfidfVectorizer(stop_words="english", max_df=0.85)
tfidf_matrix = vectorizer.fit_transform(df["combined"])

# Explanation function
def explanation_for(query, row):
    q = query.lower()
    reasons = []
    for g in row["genre"].lower().split(";"):
        if g in q: reasons.append(f"matches genre '{g}'")
    if "1990" in q and 1990 <= row["year"] <= 1999:
        reasons.append("released in the 1990s")
    for name in row["cast"].split(","):
        if name.strip().split()[0].lower() in q:
            reasons.append(f"stars {name.strip().split()[0]}")
    if not reasons: reasons.append("has similar themes to your query")
    return "Because it " + ", and it ".join(reasons) + "."

# Recommendation function
def recommend(query, top_k=5):
    q_vec = vectorizer.transform([query])
    cosine_sim = linear_kernel(q_vec, tfidf_matrix).flatten()

    # Optional decade boost
    if "1990" in query:
        decade_boost = df["year"].between(1990,1999) * 0.2
        cosine_sim += decade_boost

    top_idx = cosine_sim.argsort()[::-1][:top_k]
    results = df.iloc[top_idx].copy()
    results["score"] = cosine_sim[top_idx]
    results["explanation"] = results.apply(lambda r: explanation_for(query, r), axis=1)
    return results[["title","year","genre","cast","plot","score","explanation"]]


In [5]:
# GUI components
query_box = widgets.Text(
    value='light romantic comedies from the 1990s',
    placeholder='Type your movie request...',
    description='Query:',
    layout=widgets.Layout(width='80%')
)

button = widgets.Button(description='Recommend', button_style='success')
output = widgets.Output()

# Button click event
def on_click(b):
    with output:
        clear_output()
        query = query_box.value
        recs = recommend(query)
        display(f"Top recommendations for: \"{query}\"")
        for i, r in recs.iterrows():
            print(f"\n🎬 {r.title} ({r.year}) — {r.genre}")
            print(f"Cast: {r.cast}")
            print(f"Plot: {r.plot}")
            print(f"Why: {r.explanation} [score={r.score:.3f}]")

button.on_click(on_click)
display(query_box, button, output)


Text(value='light romantic comedies from the 1990s', description='Query:', layout=Layout(width='80%'), placeho…

Button(button_style='success', description='Recommend', style=ButtonStyle())

Output()